In [1]:
import os
import pandas as pd


In [2]:
%pwd

'c:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps\\research'

In [3]:
os.chdir(r"C:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps")


In [4]:
%pwd

'C:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps'

In [5]:
import numpy as np

In [6]:
from dataclasses import dataclass
from pathlib import Path
@dataclass

class model_evalulation_path:
    sim_matrix_path: Path
    featured_df_path :Path 
    model_eval_metrics : Path


In [7]:
from src.recommendation_system.constants import CONFIG_PATH
from src.recommendation_system.utils.common import read_yaml , create_dir
from dataclasses import dataclass
from src.recommendation_system.logging import logger
import pickle

In [8]:
class Congif_manager:

    def __init__(self, config =CONFIG_PATH):
        self.config = read_yaml(config)

        create_dir([self.config.artifacts_root])

    def get_model_evaluation(self) -> model_evalulation_path:
        config = self.config.model_evalvation

        create_dir([config.model_eval_metrics])

        return model_evalulation_path(
            sim_matrix_path        = (config.sim_matrix_path),
            featured_df_path       = (config.featured_df_path),
            model_eval_metrics = (config.model_eval_metrics)
            #mlflow_uri             = config.mlflow_uri,
            #mlflow_experiment      = config.mlflow_experiment
        ) 

In [ ]:

class mdele_valuation_config:

    def __init__(self, config):
        self.config     = config
        self.sim_matrix = None
        self.df         = None

    def load_artifacts(self):
        try:
            logger.info("=" * 20 + " Loading Artifacts STARTED " + "=" * 20)

            with open(self.config.sim_matrix_path, 'rb') as f:
                self.sim_matrix = pickle.load(f)
            with open(self.config.featured_df_path, 'rb') as f:
                self.df = pickle.load(f)

            logger.info("sim_matrix : %s", str(self.sim_matrix.shape))
            logger.info("df         : %s", str(self.df.shape))
            logger.info("=" * 20 + " Loading Artifacts COMPLETED " + "=" * 20)

        except Exception as e:
            logger.error("Load artifacts FAILED - %s", str(e))
            raise e

    def get_test_sample(self) -> pd.DataFrame:
        try:
            logger.info("=" * 20 + " Test Sample STARTED " + "=" * 20)

            stratified = (
                self.df.groupby("aesthetic", group_keys=False)
                       .apply(lambda x: x.sample(min(5, len(x)), random_state=42))
            )
            top_rated    = self.df.nlargest(10, "rating")
            bottom_rated = self.df.nsmallest(10, "rating")

            test_set = (
                pd.concat([stratified, top_rated, bottom_rated])
                  .drop_duplicates()
                  .reset_index()
            )

            logger.info("Sample size : %d", len(test_set))
            logger.info("=" * 20 + " Test Sample COMPLETED " + "=" * 20)

            return test_set

        except Exception as e:
            logger.error("Test sample FAILED - %s", str(e))
            raise e

    def precision_at_k(self, recommended_idx, relevant_idx, k) -> float:
        recommended_set = set(recommended_idx[:k])
        relevant_set    = set(relevant_idx)
        hits            = len(recommended_set & relevant_set)
        return hits / k

    def ndcg_at_k(self, recommended_idx, relevant_idx, k) -> float:
        relevant_set = set(relevant_idx)

        dcg = sum(
            1 / np.log2(i + 2)
            for i, idx in enumerate(recommended_idx[:k])
            if idx in relevant_set
        )

        ideal_hits = min(k, len(relevant_set))
        idcg       = sum(1 / np.log2(i + 2) for i in range(ideal_hits))

        return dcg / idcg if idcg > 0 else 0.0

    def score_product(self, row, top_k=10) -> dict:
        try:
            product_id = row.name
            aesthetic  = row["aesthetic"]

            # ground truth — same aesthetic, different product
            relevant_idx = self.df[
                (self.df["aesthetic"] == aesthetic) &
                (self.df.index        != product_id)
            ].index.tolist()

            # get top k from similarity matrix
            scores             = self.sim_matrix[product_id].copy()
            scores[product_id] = -1
            top_idx            = scores.argsort()[::-1][0:top_k]

            # calculate metrics
            precision = self.precision_at_k(top_idx, relevant_idx, top_k)
            ndcg      = self.ndcg_at_k(top_idx, relevant_idx, top_k)

            logger.info(
                "%-35s precision=%.2f ndcg=%.2f",
                row["product_name_clean"][:35],
                precision,
                ndcg
            )

            return {
                "product_name_clean" : row["product_name_clean"],
                "aesthetic"          : aesthetic,
                "n_relevant"         : len(relevant_idx),
                "precision_at_k"     : round(precision, 4),
                "ndcg_at_k"          : round(ndcg,      4),
            }

        except Exception as e:
            logger.error("score_product FAILED - %s", str(e))
            raise e

    def run_evaluation(self, top_k=10):
        try:
            logger.info("=" * 20 + " Evaluation STARTED " + "=" * 20)

            if self.df is None or self.sim_matrix is None:
                self.load_artifacts()

            test_sample = self.get_test_sample()

            logger.info("Scoring %d products", len(test_sample))

            records = []
            for _, row in test_sample.iterrows():
                result = self.score_product(row, top_k=top_k)
                records.append(result)

            metrics_df = pd.DataFrame(records)

            # summary
            summary = {
                "n_products_tested" : len(metrics_df),
                "top_k"             : top_k,
                "precision_at_k"    : round(metrics_df["precision_at_k"].mean(), 4),
                "ndcg_at_k"         : round(metrics_df["ndcg_at_k"].mean(),      4),
            }

            logger.info("======= SUMMARY =======")
            for k, v in summary.items():
                logger.info("%-25s : %s", k, v)

            # save
            out_path = os.path.join(
                self.config.model_eval_metrics, 'eval_metrics.csv'
            )
            metrics_df.to_csv(out_path, index=False)
            logger.info("Saved metrics -> %s", out_path)
            logger.info("=" * 20 + " Evaluation COMPLETED " + "=" * 20)

            return summary, metrics_df

        except Exception as e:
            logger.error("Evaluation FAILED - %s", str(e))
            raise e

In [ ]:
con   = Congif_manager()
model = con.get_model_evaluation()
model = ModelEvaluation(model)
model.run_evaluation()   



[2026-03-04 00:25:21,426: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-04 00:25:21,438: INFO: common: created directory at: artifacts]
[2026-03-04 00:25:21,441: INFO: common: created directory at: artifacts/model_evalvation/model/]
[2026-03-04 00:25:21,446: INFO: 2514214671: ===== Evaluation STARTED =====]
[2026-03-04 00:25:21,448: INFO: 2514214671: [ Step 1 ] Loading artifacts]


[2026-03-04 00:25:31,513: INFO: 2514214671:   sim_matrix : (13813, 13813)]
[2026-03-04 00:25:31,523: INFO: 2514214671:   df         : (13813, 14)]
[2026-03-04 00:25:31,523: INFO: 2514214671: [ Step 1 ] Done
]
[2026-03-04 00:25:31,533: INFO: 2514214671: [ Step 2 ] Building test sample]
[2026-03-04 00:25:31,773: INFO: 2514214671:   Sample size : 55]
[2026-03-04 00:25:31,773: INFO: 2514214671: [ Step 2 ] Done
]
[2026-03-04 00:25:31,773: INFO: 2514214671: [ Step 3 ] Scoring 55 products]
[2026-03-04 00:25:31,793: INFO: 2514214671:   mens cotton rich strechable knit sh precision=0.10  ndcg=0.22]
[2026-03-04 00:25:31,803: INFO: 2514214671:   mens skinny fit jeans low rise jean precision=0.20  ndcg=0.36]
[2026-03-04 00:25:31,807: INFO: 2514214671:   mens slim fit solid mid rise mid wa precision=0.10  ndcg=0.22]
[2026-03-04 00:25:31,814: INFO: 2514214671:   mens solid polo tshirt regular fit  precision=0.40  ndcg=0.50]
[2026-03-04 00:25:31,823: INFO: 2514214671:   mens solid denim jeans        

C:\Users\sam\AppData\Local\Temp\ipykernel_18316\2514214671.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(5, len(x)), random_state=42))


[2026-03-04 00:25:31,870: INFO: 2514214671:   mens super combed cotton rich regul precision=0.20  ndcg=0.21]
[2026-03-04 00:25:31,880: INFO: 2514214671:   men cotton graphic oversized fit ts precision=0.40  ndcg=0.29]
[2026-03-04 00:25:31,904: INFO: 2514214671:   mens dress smart casual shoes oxfor precision=0.10  ndcg=0.14]
[2026-03-04 00:25:31,926: INFO: 2514214671:   mens solid trackpants slimfit track precision=0.20  ndcg=0.18]
[2026-03-04 00:25:31,933: INFO: 2514214671:   men willow sneaker                  precision=0.10  ndcg=0.09]
[2026-03-04 00:25:31,950: INFO: 2514214671:   imported fabric cross lining slim f precision=0.00  ndcg=0.00]
[2026-03-04 00:25:31,957: INFO: 2514214671:   mens regular fit blazer             precision=0.00  ndcg=0.00]
[2026-03-04 00:25:31,969: INFO: 2514214671:   cotton polyester blend solid casual precision=0.00  ndcg=0.00]
[2026-03-04 00:25:31,977: INFO: 2514214671:   men comfort slim fit pure cotton ta precision=0.00  ndcg=0.00]
[2026-03-04 00:25:3

Traceback (most recent call last):
  File "c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\sam\AppData\Local\Temp\ipykernel_18316\1399911550.py", line 4, in <module>
    model.run_evaluation()
  File "C:\Users\sam\AppData\Local\Temp\ipykernel_18316\2514214671.py", line 144, in run_evaluation
    metrics_df.to_csv(out / "eval_metrics.csv", index=False)
                      ~~~~^~~~~~~~~~~~~~~~~~~~
TypeError: unsupported operand type(s) for /: 'str' and 'str'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 2201, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  

In [29]:
import pandas as pd 
df = pd.read_csv(r'artifacts\data_transformation\transformed data\Transformed_Data.csv')

In [15]:
import pickle

with open(r'artifacts\feature_engineering\model\sim_matrix.pkl', 'rb') as f:
    sim_matrix = pickle.load(f)
    sim_matrix

In [20]:
sim_matrix[1]

array([0.15691217, 1.        , 0.09687231, ..., 0.        , 0.        ,
       0.        ], shape=(13813,))

In [27]:
sim_matrix[54].argsort()[::-1][0:10]

array([  54, 2114, 8453, 2172,  106, 8440,  163, 8580,  258, 8539])

In [73]:
df[(df["aesthetic"] == 'streetwear')].index[1::][1:10]

Index([2, 3, 4, 5, 6, 7, 8, 9, 10], dtype='int64')

In [72]:
sim_matrix[1].argsort()[::-1][1:10]

array([8421, 2095,    1,   43, 8428,    5, 8426, 7354, 8435])

In [65]:
relevant_idx = df[
    (df["aesthetic"] == aesthetic) &
    (df.index != product_id)
].index.tolist()

In [66]:
print("relevant_idx count :", len(relevant_idx))
print("first 10           :", relevant_idx[:10])

relevant_idx count : 2581
first 10           : [3946, 3947, 3948, 3949, 3950, 3951, 3952, 3953, 3954, 3955]


In [59]:
df.loc[2000]

asin                                                     B0D1XXW98D
product_name      Unisex Regular Cotton Combo of 3 Solid Color T...
price                                                      6.809039
rating                                                          4.2
review_count                                               6.888572
discount                                                         55
image_url         https://m.media-amazon.com/images/I/41ITY79hWU...
product_link      https://www.amazon.in/XENOVAURBAN-Unisex-Regul...
aesthetic                                            college_casual
is_new_product                                                    0
Name: 2000, dtype: object